# 01 · Quick interview topics

Twelve topics that come up again and again, one screen each. For every topic you get what it is in plain English, the 30-second interview answer, a small demo you can run, and a link to the notebook that goes deep.

**Time:** ~45 min · **Runs:** offline on CPU in about a minute (it loads two small cached models, a reranker and an embedder, about 90 MB each) · **Needs:** [00 · How to use these notebooks](00_how_to_use_these_notebooks.ipynb)

## Why this matters in interviews

- These topics show whether you have *used* the tools. One number ("p95", "4× smaller", "128 KiB per token") or one line of code beats a paragraph of theory.
- Several are trap topics: the mean hides p95, a reranker cannot fix recall, the global `np.random.seed`, and Power BI's silent de-duplication. A precise answer stands out.
- Each answer here is deliberately short. The linked deep dive gives you the depth for the follow-up question.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `po12` | How would you build a model router, and what decides when to escalate? |
| `tn28` | What is overfitting in the context of fine-tuning an LLM, and how do you spot it? |
| `fo30` | What is quantisation, in one paragraph, and what does it cost you? |
| `pd13` | What is a fallback model strategy and how do you test it? |
| `rg08` | What is reranking, and why can a reranker not fix bad recall? |
| `ve08` | What is BM25 and how does it work? |
| `in02` | Explain the KV cache: what it stores, why it works, and what it costs. |
| `rg42` | What is semantic chunking and is it worth the cost? |
| `tl20` | What does Pydantic do for an LLM application? |
| `pd41` | What is the difference between p50, p95 and p99, and which do you optimise? |

## Contents

| # | Topic | In one line |
|---|---|---|
| 1 | [Jev & Laya: System-1 decision models](#t1) | typed questions in, probabilities out, no generated text |
| 2 | [Power BI-style fast charts in Python](#t2) | `df.plot`, `pivot_table`, and the `dataset` convention |
| 3 | [Overfitting](#t3) | training error falls while validation error rises |
| 4 | [Quantization](#t4) | fewer bits per weight: int8 is 4× smaller than fp32 |
| 5 | [LiteLLM](#t5) | one `completion()` for 100+ providers, with mocks and fallbacks |
| 6 | [Re-ranking](#t6) | cheap top 20, then a cross-encoder reorders them |
| 7 | [BM25](#t7) | keyword scoring that nails exact IDs and error codes |
| 8 | [KV cache](#t8) | reuse past keys and values, trading memory for speed |
| 9 | [Semantic chunking](#t9) | cut where consecutive sentences stop being similar |
| 10 | [Pydantic](#t10) | validated data, and the JSON Schema that structured outputs use |
| 11 | [P95 latency](#t11) | the tail users feel, which the mean hides |
| 12 | [`np.random.seed` vs `default_rng`](#t12) | reproducible randomness, the modern way |

In [ ]:
import os
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE, AQUA, GRAY, INK = "#2a78d6", "#eb6834", "#1baf7a", "#b5b3ab", "#52514e"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.titlesize": 11,
    "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]),
})
print("numpy", np.__version__)

<a id="t1"></a>
## 1 · Jev & Laya: "System 1" decision models

**In plain English.** Most "decisions" inside an agent or a support tool are really classification. Which team gets this ticket? Is it urgent? Is this tool call safe? A chat LLM makes you wait seconds, pay for tokens, and parse free text that might not even be one of your labels. A **System 1 model** is named after Kahneman's fast, intuitive thinking. It takes a *state* plus *typed questions* and, in one fast pass, returns a probability for every possible answer. Nothing is generated, so there is nothing to parse and no invented labels.

| | **Jev** (TypeSafe AI) | **Laya** (ConvAI Innovations) |
|---|---|---|
| What | hosted "System One" model, API only | open-source local alternative |
| Get it | early access from 15 Sep 2026 (waitlist); `pip install typesafe-sdk` | `pip install laya`; Apache-2.0 weights on Hugging Face |
| Runs on | TypeSafe's cloud; key in `TYPESAFE_API_KEY` | your CPU or GPU; ModernBERT-based checkpoints (`convaiinnovations/laya`, a multilingual one, ...) |
| Question types | `Choice` (up to 255 options), `Score` (2–10 ordered levels), `Noul` (yes/no, gives P(true)) | the same three: `"choice"`, `"score"`, `"noul"` |
| Many questions per request | yes: they don't see each other, so the state is read once | yes |
| Speed / cost (vendor-reported) | 70–500 ms end to end; roughly 200× faster and 400× cheaper than a frontier LLM on classification-type decisions | about 40 ms per question on a T4 GPU, less when batched |
| Integrations | LangChain: `pip install --pre "langchain-typesafe[experimental]"` (pre-release), then `TypeSafeClassifier` | `laya.serve` speaks the same `/v1/systemone` wire protocol |

```
ticket --> System 1 (typed questions) --> probabilities --> confidence >= threshold ? --yes--> act: route, tag, run the tool
                                                                                   '--no---> System 2: an LLM or a human
```

> **30-second answer.** "For classification-shaped decisions inside an agent (routing, urgency, 'is this safe?') I don't generate text with a chat LLM. I use a System 1 decision model: TypeSafe's Jev if hosted is fine, or Laya if it must run locally. I send the state plus typed questions (choice, score, yes/no) in one request and get calibrated probabilities back in milliseconds, with no parsing and no hallucinated labels. My code branches on the probability, and anything below a confidence threshold escalates to a System 2 LLM or a human. The threshold rises with the stakes of the action."

**Deep dive:** [System-1 decision models: Jev & Laya](../15_agentic_ai/08_system1_decision_models_jev_and_laya.ipynb)

**What goes over the wire.** This is an abridged request to `POST /v1/systemone`, with a response whose numbers are illustrative:

```json
{"model": "jev-latest",
 "state": "Hi, I was charged twice this month and need one payment refunded today.",
 "questions": {
   "department": {"type": "choice", "instructions": "Which team should handle this?",
                  "criteria": {"billing": "invoices, charges, refunds",
                               "technical": "bugs, errors, outages",
                               "account": "login, password, profile"}},
   "urgent":     {"type": "noul",  "instructions": "The customer needs help right now"},
   "frustration":{"type": "score", "instructions": "How frustrated is the customer?",
                  "criteria": ["calm", "annoyed", "angry"]}}}
```
```json
{"answers": {
   "department":  {"type": "choice", "choice": "billing", "confidence": 0.97,
                   "probabilities": {"billing": 0.97, "technical": 0.01, "account": 0.02}},
   "urgent":      {"type": "noul", "noul": 0.81},
   "frustration": {"type": "score", "score": 1.28, "probabilities": [0.10, 0.52, 0.38]}}}
```

The SDKs, as published by the vendors in September 2026. They are shown as text because they need the packages, and Jev needs an early-access key. Early-access APIs change, so check the current docs:

```python
# Jev (hosted): pip install typesafe-sdk, key in TYPESAFE_API_KEY
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

client = TypeSafeClient()
response = client.system_one(
    state={"ticket": ticket_text},
    questions={
        "department": Choice(instructions="Which team should handle this?",
                             criteria={"billing": "invoices, charges, refunds", "technical": "bugs, errors", "account": "login, profile"}),
        "urgent": Noul(instructions="The customer needs help right now"),
        "frustration": Score(instructions="How frustrated is the customer?", criteria=["calm", "annoyed", "angry"]),
    },
)
response.answers["department"].choice, response.answers["department"].confidence, response.answers["urgent"].noul
```
```python
# Laya (local, Apache-2.0): pip install laya; downloads a ~400M-parameter checkpoint on first use
from laya import Router

router = Router(preload=True)                      # load the checkpoints up front; Router() loads lazily
result = router.predict(ticket_text, {
    "department": {"type": "choice", "instructions": "Which team should handle this?",
                   "criteria": {"billing": "invoices, charges, refunds", "technical": "bugs, errors", "account": "login, profile"}},
    "urgent": {"type": "noul", "instructions": "The customer needs help right now"},
})
result["answers"]["department"]["choice"], result["answers"]["urgent"]["noul"], result["routing"]["model"]
```
```python
# LangChain: pip install langchain-typesafe
from langchain_typesafe import Noul, TypeSafeClassifier

response = TypeSafeClassifier().invoke({"state": ticket_text,
                                        "questions": {"urgent": Noul(instructions="Does this need attention right now?")}})
response.nouls["urgent"].noul
```

**A local stand-in you can run.** It has the same contract, a state plus typed questions in and probabilities out, but it is built from TF-IDF features and logistic regression trained on 24 inline tickets. It is *not* Jev. Its probabilities come from 24 examples, so they are rough rather than calibrated. What it shows is the pattern: your code branches on a probability, not on parsed text.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

TICKETS = [  # (text, department, urgent?)
    ("I was charged twice for my subscription this month", "billing", 0),
    ("Please send me the invoice for August", "billing", 0),
    ("My card was declined but the payment still went through", "billing", 0),
    ("Refund my last payment, I cancelled before the renewal", "billing", 0),
    ("You billed my company card five times today, stop charging it now", "billing", 1),
    ("How do I change the billing address on my invoices?", "billing", 0),
    ("We are overcharged every hour and our budget is gone, fix this immediately", "billing", 1),
    ("Can I pay annually instead of monthly?", "billing", 0),
    ("The app crashes when I upload a PDF", "technical", 0),
    ("Production is down, all API calls return 500 errors", "technical", 1),
    ("The dashboard is slow to load in the afternoon", "technical", 0),
    ("Our checkout is broken and customers cannot pay, urgent", "technical", 1),
    ("Export to CSV produces an empty file", "technical", 0),
    ("Webhooks stopped firing an hour ago and orders are stuck", "technical", 1),
    ("Dark mode makes the chart labels hard to read", "technical", 0),
    ("The API returns timeout errors for every request since the deploy", "technical", 1),
    ("How do I reset my password?", "account", 0),
    ("I am locked out of my account and have a client demo in ten minutes", "account", 1),
    ("Please change the email address on my profile", "account", 0),
    ("Someone logged into my account from another country, lock it now", "account", 1),
    ("How do I add a teammate to my workspace?", "account", 0),
    ("Delete my account and all my data", "account", 0),
    ("Two-factor codes are not arriving and I cannot log in during an outage", "account", 1),
    ("Can I rename my organisation?", "account", 0),
]
texts = [t for t, _, _ in TICKETS]
new_head = lambda: make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                                 LogisticRegression(C=10, max_iter=2000))
HEADS = {"department": new_head().fit(texts, [d for _, d, _ in TICKETS]),     # one small model per question
         "urgent": new_head().fit(texts, [u for _, _, u in TICKETS])}

def system_one(state: str, questions: dict) -> dict:
    """Jev/Laya-shaped response: {"answers": {name: {...}}} with probabilities for every answer."""
    answers = {}
    for name, q in questions.items():
        probs = dict(zip(HEADS[name].classes_.tolist(), HEADS[name].predict_proba([state])[0].tolist()))
        if q["type"] == "choice":
            best = max(probs, key=probs.get)
            answers[name] = {"type": "choice", "choice": best, "confidence": probs[best], "probabilities": probs}
        elif q["type"] == "noul":
            answers[name] = {"type": "noul", "noul": probs[1]}
    return {"answers": answers}

In [ ]:
import json

QUESTIONS = {
    "department": {"type": "choice", "instructions": "Which team should handle this?",
                   "criteria": {"billing": "charges, refunds", "technical": "bugs, outages", "account": "login, profile"}},
    "urgent": {"type": "noul", "instructions": "The customer needs help right now"},
}
NEW = ["I think I was billed twice, can you refund one payment?",
       "The API is down and every request fails, customers are affected right now",
       "How can I change my password?",
       "My invoice looks wrong and the app also crashes"]
THRESHOLD = 0.60                  # below this we don't act; raise it for risky actions

def rounded(obj, nd=3):
    """A copy with floats rounded, for printing."""
    if isinstance(obj, float):
        return round(obj, nd)
    return {k: rounded(v, nd) for k, v in obj.items()} if isinstance(obj, dict) else obj

t0 = time.perf_counter()
results = [system_one(t, QUESTIONS) for t in NEW]
ms = (time.perf_counter() - t0) * 1000 / len(NEW)
print("response for ticket 1:", json.dumps(rounded(results[0]), indent=1), "\n")

for text, r in zip(NEW, results):
    d, p_urgent = r["answers"]["department"], r["answers"]["urgent"]["noul"]
    route = f"-> {d['choice']} queue" if d["confidence"] >= THRESHOLD else "-> ESCALATE to an LLM"
    print(f"{text[:46]:47} {d['choice']:9} conf={d['confidence']:.2f}  P(urgent)={p_urgent:.2f}  {route}")
print(f"\n{ms:.1f} ms per ticket for both questions")

for r in results:
    probs = r["answers"]["department"]["probabilities"]
    assert abs(sum(probs.values()) - 1) < 1e-9 and r["answers"]["department"]["choice"] == max(probs, key=probs.get)
assert results[0]["answers"]["department"]["choice"] == "billing"
assert results[1]["answers"]["urgent"]["noul"] > 0.5 > results[2]["answers"]["urgent"]["noul"]
assert results[3]["answers"]["department"]["confidence"] < THRESHOLD      # the ambiguous ticket escalates

The ambiguous ticket (an invoice problem *and* a crash) spreads its probability across teams, so the code escalates instead of guessing. That is the whole design: **System 1 for the confident bulk, System 2 for the rest.** A chat-LLM call typically takes 0.5–3 s, while the stand-in above takes about a millisecond. The real models trade a little of that speed for far better accuracy and calibration.

<a id="t2"></a>
## 2 · Power BI-style fast charts, in Python

**In plain English.** "Fast chart creation" means going from a table to a chart in one line. In pandas that is `df.plot.bar(...)`, or `pivot_table(...)` followed by `.plot()`. The same skill carries straight into Power BI. A **Python visual** runs your script with the fields you dragged in, handed to you as a pandas DataFrame named **`dataset`**. You draw with matplotlib, and Power BI shows whatever `plt.show()` renders. Power BI writes this preamble above your code:

```python
# The following code to create a dataframe and remove duplicated rows is always executed and acts as a preamble for your script:
# dataset = pandas.DataFrame(region, product, revenue)
# dataset = dataset.drop_duplicates()
```

- **Rows are de-duplicated.** Two identical sales rows become one and your totals shrink. Add an ID field to keep every row.
- The output is a **static image**. It reacts to slicers and filters, but you cannot click it to cross-filter other visuals.
- Limits: at most 150,000 rows are passed in, and scripts time out after about 5 minutes. Power BI Desktop needs a local Python with pandas and matplotlib.

> **30-second answer.** "For quick charts I aggregate in pandas and plot in one line, for example `df.groupby('region')['revenue'].sum().plot.barh()`, or `pivot_table(index='month', columns='region', values='revenue', aggfunc='sum').plot()`. In Power BI a Python visual gets a de-duplicated DataFrame called `dataset` with the selected fields. I draw with matplotlib and end with `plt.show()`, and I include an ID field so genuine duplicate rows aren't merged. For interactivity and cross-filtering I use native visuals. Python visuals are for charts Power BI can't draw."

**Deep dive:** [Matplotlib and quick charts](../04_numpy_pandas_pytorch/04_matplotlib_and_quick_charts.ipynb)

In [ ]:
import pandas as pd

rng = np.random.default_rng(2)
n = 240
sales = pd.DataFrame({
    "order_id": np.arange(1, n + 1),
    "month": rng.choice([f"2026-{m:02d}" for m in range(1, 7)], n),
    "region": rng.choice(["North", "South", "East", "West"], n, p=[0.35, 0.25, 0.25, 0.15]),
    "product": rng.choice(["Basic", "Pro", "Team"], n, p=[0.5, 0.3, 0.2]),
})
sales["revenue"] = sales["product"].map({"Basic": 50, "Pro": 150, "Team": 400}) * rng.integers(1, 4, n)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
sales.groupby("region")["revenue"].sum().sort_values().plot.barh(ax=axes[0], color=BLUE, title="Revenue by region: one line")
monthly = sales.pivot_table(index="month", columns="region", values="revenue", aggfunc="sum")
monthly.plot(ax=axes[1], marker="o", title="Monthly revenue by region: pivot_table, then .plot()")
axes[1].set_ylim(0, monthly.to_numpy().max() * 1.3)
axes[1].legend(ncol=4, fontsize=8, loc="upper center", frameon=False)
axes[0].set_xlabel("revenue"); axes[1].set_ylabel("revenue")
plt.tight_layout(); plt.show()

In [ ]:
# What Power BI does before your script runs, with fields region, product, revenue dragged in:
dataset = pd.DataFrame(sales[["region", "product", "revenue"]]).drop_duplicates()
print(f"rows in the table     : {len(sales)}")
print(f"rows your script gets : {len(dataset)}   (identical rows were merged)")
print(f"revenue in the table  : {sales['revenue'].sum():,}")
print(f"revenue in `dataset`  : {dataset['revenue'].sum():,}   <- wrong total")
assert dataset["revenue"].sum() < sales["revenue"].sum()

# The fix: add an ID field, so no two rows are identical. Then the script body, exactly as you'd paste it:
dataset = pd.DataFrame(sales[["order_id", "region", "product", "revenue"]]).drop_duplicates()
assert dataset["revenue"].sum() == sales["revenue"].sum()

pivot = dataset.pivot_table(index="region", columns="product", values="revenue", aggfunc="sum")
ax = pivot.plot.bar(figsize=(8, 3.3), rot=0, title="Power BI Python visual: revenue by region and product")
ax.set_ylabel("revenue")
plt.show()

<a id="t3"></a>
## 3 · Overfitting

**In plain English.** The model memorises the training data, noise included, instead of learning the pattern. It looks brilliant on data it has seen and disappoints on new data. **Spot it** when training error keeps falling while validation error stops falling and rises. That widening gap is the signature. **Fix it** with more or cleaner data, a simpler model, regularisation (L2 / weight decay, dropout), early stopping at the best validation score, and cross-validation to choose the complexity. Fine-tuning an LLM shows the same pattern: eval loss turns up after an epoch or two while train loss keeps dropping. The remedies are fewer epochs, a lower learning rate or a smaller LoRA rank, and more varied data.

> **30-second answer.** "Overfitting is when a model fits the noise: low training error, high validation error. I spot it on a learning curve, where train loss keeps dropping while validation loss bottoms out and climbs. I fix it with more data, a simpler model, regularisation such as weight decay and dropout, early stopping on the validation metric, and cross-validation to pick the complexity. When fine-tuning an LLM I watch eval loss every epoch and keep the checkpoint at the minimum."

**Deep dive:** [Overfitting, regularization & cross-validation](../05_machine_learning/05_overfitting_regularization_cross_validation.ipynb)

In [ ]:
from numpy.polynomial import Polynomial

rng = np.random.default_rng(3)
true_fn = lambda x: np.sin(2 * np.pi * x)
x_tr = np.sort(rng.uniform(0, 1, 20));  y_tr = true_fn(x_tr) + rng.normal(0, 0.25, 20)
x_va = np.sort(rng.uniform(0, 1, 200)); y_va = true_fn(x_va) + rng.normal(0, 0.25, 200)
mse = lambda p, x, y: float(np.mean((p(x) - y) ** 2))
fit = lambda deg: Polynomial.fit(x_tr, y_tr, deg)          # least squares on a rescaled x: well conditioned

grid = np.linspace(0, 1, 400)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.3), sharey=True)
for ax, deg, verdict in zip(axes, [1, 4, 15], ["underfit", "good fit", "overfit"]):
    p = fit(deg)
    ax.plot(grid, true_fn(grid), color=GRAY, lw=1.5, label="true pattern")
    ax.plot(grid, p(grid), color=BLUE, lw=2, label="model")
    ax.scatter(x_tr, y_tr, s=16, color=INK, zorder=3, label="training points")
    ax.set_ylim(-2, 2)
    ax.set_title(f"degree {deg}: {verdict}\ntrain MSE {mse(p, x_tr, y_tr):.3g} · validation MSE {mse(p, x_va, y_va):.3g}", fontsize=9)
axes[0].legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

In [ ]:
degrees = list(range(1, 16))
train_err = [mse(fit(d), x_tr, y_tr) for d in degrees]
val_err = [mse(fit(d), x_va, y_va) for d in degrees]
best = degrees[int(np.argmin(val_err))]

fig, ax = plt.subplots(figsize=(8.5, 3.4))
ax.plot(degrees, train_err, marker="o", label="training error")
ax.plot(degrees, val_err, marker="o", label="validation error")
ax.axvline(best, color=GRAY, lw=1)
ax.text(best + 0.2, max(val_err) / 4, f"best degree by validation = {best}", fontsize=9, color=INK)
ax.set_yscale("log"); ax.set_xlabel("polynomial degree (model complexity)"); ax.set_ylabel("MSE (log scale)")
ax.set_title("Training error keeps falling; validation error turns up. The gap is overfitting.")
ax.legend(); plt.show()

assert train_err[14] < train_err[3]      # degree 15 fits the training points better than degree 4 ...
assert val_err[14] > val_err[3]          # ... and does worse on new data
print(f"degree 4 : train {train_err[3]:.3f}  validation {val_err[3]:.3g}")
print(f"degree 15: train {train_err[14]:.3f}  validation {val_err[14]:.3g}  <- it swings wildly between and beyond the points")

<a id="t4"></a>
## 4 · Quantization

**In plain English.** Store each weight with fewer bits: fp32 uses 4 bytes, fp16/bf16 2, int8 1, and int4 half a byte. A 7B model shrinks from 28 GB to 3.5 GB of weights, so it fits smaller GPUs and runs faster, because decoding is limited by memory bandwidth. The cost is a little precision. **absmax int8:** take `scale = max|w| / 127`, store `q = round(w / scale)` as int8, and rebuild `w ≈ q × scale`. A single outlier inflates the scale for everything, which is why real schemes use per-channel or per-group scales, keep outliers in higher precision (LLM.int8()), or use NF4 levels shaped like the weight distribution (QLoRA).

> **30-second answer.** "Quantization stores weights in fewer bits. int8 is 4× smaller than fp32 and int4 is 8× smaller, so a 7B model goes from 28 GB to about 3.5 GB. The basic recipe is absmax: scale so the largest weight maps to 127, then round. The cost is rounding error, and outliers make it worse, so production methods use per-channel or group-wise scales, outlier handling, or NF4 (bitsandbytes), GPTQ or AWQ for 4-bit. int8 is close to lossless. 4-bit usually costs a little quality, so I check it on my own evals before shipping."

**Deep dives:** [Quantization int8 / int4 / NF4](../07_genai_foundations/07_quantization_int8_int4_nf4.ipynb) · [Quantization with bitsandbytes](../10_huggingface/06_quantization_with_bitsandbytes.ipynb)

In [ ]:
rng = np.random.default_rng(4)
W = rng.normal(0, 0.02, size=(1024, 1024)).astype(np.float32)     # like one weight matrix of a transformer layer

def absmax_int8(w, axis=None):
    """One scale per tensor (axis=None) or one per row (axis=1)."""
    scale = np.abs(w).max(axis=axis, keepdims=axis is not None) / 127.0
    return np.round(w / scale).astype(np.int8), np.float32(scale)

rel_err = lambda w, q, s: float(np.linalg.norm(w - q.astype(np.float32) * s) / np.linalg.norm(w))

q, s = absmax_int8(W)
print(f"fp32 {W.nbytes / 2**20:.1f} MiB -> int8 {q.nbytes / 2**20:.2f} MiB ({W.nbytes // q.nbytes}x smaller, plus one scale)")
print(f"relative reconstruction error: {rel_err(W, q, s):.2%}")
assert W.nbytes == 4 * q.nbytes and rel_err(W, q, s) < 0.02

W_out = W.copy(); W_out[0, 0] = 1.0                                 # one outlier, 50x a typical weight
err = {name: rel_err(W_out, *absmax_int8(W_out, axis)) for name, axis in [("per-tensor", None), ("per-row", 1)]}
for name, e in err.items():
    print(f"with one outlier, {name:10} scale: error {e:.2%}")
assert err["per-row"] < err["per-tensor"]                           # finer-grained scales contain the damage

In [ ]:
bytes_per_weight = {"fp32": 4, "fp16 / bf16": 2, "int8": 1, "int4": 0.5}
size_7b = {k: 7e9 * b / 1e9 for k, b in bytes_per_weight.items()}
assert size_7b["fp32"] == 28 and size_7b["int4"] == 3.5

fig, ax = plt.subplots(figsize=(8.5, 2.8))
bars = ax.barh(list(size_7b)[::-1], list(size_7b.values())[::-1], color=BLUE, height=0.55)
ax.bar_label(bars, fmt="%.1f GB", padding=3, fontsize=9)
ax.axvline(24, color=INK, lw=1)
ax.text(24.4, 1.5, "24 GB\nconsumer GPU", fontsize=8, color=INK, va="center")
ax.set_xlim(0, 33); ax.set_xlabel("GB of weights (the KV cache and activations come on top)")
ax.set_title("A 7B model's weights by precision: fp32 won't fit a 24 GB GPU, int4 fits a laptop GPU")
plt.show()

<a id="t5"></a>
## 5 · LiteLLM

**In plain English.** Every provider has its own SDK and response format. LiteLLM gives you one function, `litellm.completion()`, that takes OpenAI-style `messages` and returns an OpenAI-shaped response for 100+ providers. The prefix of the model string picks the provider: `"gpt-4.1-mini"`, `"anthropic/<model>"`, `"ollama/qwen2.5:3b"`, `"bedrock/<model>"`. On top of that you get a `Router` with retries, **fallbacks** and load balancing, cost tracking (`completion_cost`), and a proxy server that turns it into a company-wide gateway with virtual keys and budgets. For tests, **`mock_response=`** returns a fake reply with no network call and no key.

> **30-second answer.** "LiteLLM is an adapter and gateway: one `completion()` call in the OpenAI format for any provider, chosen by the model string. I use its Router for fallbacks (if the primary errors or times out, try the next deployment), retries and load balancing, and `completion_cost` or the proxy for spend tracking and budgets. In CI I pass `mock_response`, including a mocked exception, so the fallback path is tested without any network. The trade-offs are one more dependency in the hot path, and provider-specific features that can lag behind the native SDK."

**Deep dive:** [LiteLLM gateway & model routing](../16_production/06_litellm_gateway_and_model_routing.ipynb)

In [ ]:
import logging

os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")   # use the bundled price table: no network at import
import litellm

litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)
logging.getLogger("LiteLLM Router").setLevel(logging.CRITICAL)

resp = litellm.completion(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "Say hello in five words."}],
    mock_response="Hello there, nice meeting you!",          # no key, no network: made for tests
)
print("type   :", type(resp).__name__, "- OpenAI-shaped, so .choices[0].message.content works")
print("content:", resp.choices[0].message.content)
print("usage  :", resp.usage.prompt_tokens, "in /", resp.usage.completion_tokens, "out (mocked counts)")
print(f"cost   : ${litellm.completion_cost(completion_response=resp):.6f} from LiteLLM's price table")
assert resp.choices[0].message.content == "Hello there, nice meeting you!"

In [ ]:
from litellm import Router

router = Router(
    model_list=[   # two deployments; the primary is mocked to fail like an outage
        {"model_name": "primary", "litellm_params": {"model": "gpt-4.1-mini", "mock_response": Exception("provider outage")}},
        {"model_name": "backup", "litellm_params": {"model": "ollama/qwen2.5:3b", "mock_response": "Answer from the backup model."}},
    ],
    fallbacks=[{"primary": ["backup"]}],
    num_retries=0,
)
r = router.completion(model="primary", messages=[{"role": "user", "content": "Hi"}])
print("asked for 'primary' (mocked outage) -> got:", r.choices[0].message.content)
print("served by:", r.model)
assert r.choices[0].message.content == "Answer from the backup model."

<a id="t6"></a>
## 6 · Re-ranking

**In plain English.** First-stage retrieval (BM25 or vector search) scores the query and each document *separately*. That lets it search millions of documents in milliseconds, but only roughly. A **cross-encoder reranker** reads the query and one document *together* through a transformer, so it sees how their words relate: "charged two times" means "billed twice". It is much more accurate, but it needs one forward pass per pair, so you can only afford it on a short list. The standard pattern is to retrieve the top 20–100 cheaply, rerank those, and send the best 3–5 to the LLM. **A reranker cannot fix bad recall.** It only reorders what the first stage found.

```
query --> BM25 / vectors (millions of docs, ms) --> top 20 --> cross-encoder (20 pairs) --> top 3 --> LLM
```

> **30-second answer.** "Retrieval is two-stage. BM25 or a bi-encoder gets high recall cheaply, say the top 50. Then a cross-encoder scores each (query, document) pair jointly and reorders them for precision, and the top few go into the prompt. Cross-encoders are too slow for first-stage search because nothing can be precomputed per document: every query costs N forward passes. They improve precision, not recall. If the right chunk isn't among the candidates, reranking can't bring it back, so I measure first-stage recall@k separately."

**Deep dive:** [Reranking](../09_rag/05_reranking.ipynb)

In [ ]:
from rank_bm25 import BM25Okapi

STOP = set("a an the is are was were be to of and or for in on at by it its this that my your our we you i "
           "how do does what can with from".split())

def tokenize(text):
    """Lowercase words; keep hyphenated codes like ERR-4042 as ONE token; drop stopwords."""
    return [w for w in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower()) if w not in STOP]

KB = [
    "Billed twice for the same month? We refund the duplicate payment and the money is back on your card within 5 business days.",
    "Money back guarantee: get your money back if you cancel an annual plan within 30 days.",
    "Back to school offer: get two months free when you sign up this month.",
    "Get paid faster: money from your sales reaches your bank within two days.",
    "You can get a copy of any invoice from this month or earlier under Billing > Invoices.",
    "Two-factor login: if you get the code two times, use the most recent one.",
    "Keep the card reader charged; a full charge lasts about two days.",
    "Month-end reports are emailed on the first business day of each month.",
    "How do I get back into my account? Use the reset link on the login page.",
    "Charged a late fee? Late fees apply when an invoice is unpaid for 14 days.",
    "Times shown in reports use your workspace time zone.",
    "Back up your data: exports run every night at 2 am.",
    "You can pay by card, bank transfer or PayPal.",
    "Change your plan at any time; the new price starts next month.",
    "Our support team answers within two hours on weekdays.",
    "Discounts: students get 50% off the monthly plan.",
    "Upload limits: free plans accept files up to 10 MB.",
    "Delete a workspace from Settings > Advanced; this cannot be undone.",
    "Currency: prices are charged in USD unless you choose EUR at checkout.",
    "Invite teammates from Settings > Members.",
    "Get notified: turn on email alerts for failed payments.",
    "API rate limits: 60 requests per minute per key.",
    "The mobile app works offline and syncs when you are back online.",
    "Taxes: VAT is added to invoices for EU customers.",
    "How do I get a receipt? Receipts are emailed after every payment.",
    "Trial accounts are not charged until the trial ends.",
]
RIGHT = 0                                   # the document that actually answers the question
query = "I was charged two times this month, how do I get my money back?"

bm25_kb = BM25Okapi([tokenize(d) for d in KB])
first_stage = np.argsort(-bm25_kb.get_scores(tokenize(query)), kind="stable")[:20]     # cheap: top 20
print("BM25 top 5 (* = the right answer):")
for rank, i in enumerate(first_stage[:5], 1):
    print(f"  {rank}. {'*' if i == RIGHT else ' '} {KB[i][:85]}")

In [ ]:
from sentence_transformers import CrossEncoder

def load_hf(cls, name):
    """Load from the local Hugging Face cache without network checks; download (~90 MB) only on first use."""
    try:
        return cls(name, local_files_only=True)
    except Exception:
        try:
            return cls(name)
        except Exception as e:
            print(f"Could not load {name} ({type(e).__name__}). Connect to the internet once to download it.")
            return None

reranker = load_hf(CrossEncoder, "cross-encoder/ms-marco-MiniLM-L6-v2")

if reranker is not None:
    t0 = time.perf_counter()
    ce_scores = reranker.predict([(query, KB[i]) for i in first_stage])      # 20 (query, doc) pairs, read jointly
    reranked = first_stage[np.argsort(-ce_scores, kind="stable")]
    print(f"reranked 20 pairs in {time.perf_counter() - t0:.2f}s. Cross-encoder top 3:")
    for rank, i in enumerate(reranked[:3], 1):
        print(f"  {rank}. {'*' if i == RIGHT else ' '} {KB[i][:85]}")
    before, after = list(first_stage).index(RIGHT) + 1, list(reranked).index(RIGHT) + 1
    print(f"\nthe right answer moved from BM25 rank {before} to rank {after}")
    assert after == 1 and before > after

In [ ]:
if reranker is not None:
    after_rank = {doc: r for r, doc in enumerate(reranked, 1)}
    fig, ax = plt.subplots(figsize=(10, 4.2))
    for r_before, doc in enumerate(first_stage[:8], 1):
        right = doc == RIGHT
        ax.plot([0, 1], [r_before, after_rank[doc]], marker="o", ms=7, lw=2.5 if right else 1,
                color=BLUE if right else GRAY, zorder=3 if right else 2)
        ax.text(-0.04, r_before, KB[doc][:44] + "...", ha="right", va="center", fontsize=8,
                color=INK if right else "#898781", fontweight="bold" if right else "normal")
    ax.set_xlim(-1.25, 1.1); ax.set_ylim(20.8, 0.2)
    ax.set_xticks([0, 1], ["BM25 rank", "cross-encoder rank"]); ax.set_yticks([1, 5, 10, 15, 20])
    ax.grid(False); ax.spines[["left", "bottom"]].set_visible(False)
    ax.set_title("Reranking BM25's top 8: the right answer (blue) jumps to #1, keyword look-alikes sink")
    plt.show()

<a id="t7"></a>
## 7 · BM25

**In plain English.** BM25 is the classic keyword-relevance score behind Elasticsearch, OpenSearch and Lucene. For every query word found in a document it combines three effects:
1. **Rare words count more.** This is IDF: `err-4042` in 1 document out of 1,000 is decisive, while "the" says nothing.
2. **Repeats help, with diminishing returns.** Term-frequency saturation is controlled by `k1`, usually 1.2 to 2.
3. **Long documents are discounted.** Length normalisation is controlled by `b`, usually 0.75.

$$\text{score}(q,d)=\sum_{t\in q}\text{IDF}(t)\cdot\frac{f(t,d)\,(k_1+1)}{f(t,d)+k_1\left(1-b+b\,\frac{|d|}{\text{avgdl}}\right)}$$

**Why it catches exact IDs and error codes.** To BM25, `ERR-4042` is just a rare token that either matches or doesn't. An embedding model sees `ERR-4041`, `ERR-4042` and `ERR-4043` as nearly the same string and places them almost on top of each other. That is why production RAG runs **hybrid** search (BM25 plus vectors, fused with RRF). The catch is that BM25 is only as good as its **tokenizer** (the "analyzer"): split badly and the code never matches.

> **30-second answer.** "BM25 sums, over the query terms, IDF times a saturating term frequency with length normalisation (k1 around 1.2 to 2, b = 0.75). It's fast, needs no training, and is excellent at exact matches such as product IDs, error codes and names, which dense embeddings blur. It misses synonyms and paraphrases, which dense retrieval catches. So I run both and fuse them with reciprocal rank fusion. The analyzer (lowercasing, punctuation, stemming) decides what can match at all, so I test it on real identifiers."

**Deep dive:** [BM25 from scratch](../09_rag/02_bm25_from_scratch.ipynb)

In [ ]:
DOCS = [
    "ERR-4041: the file name contains characters we do not support.",
    "ERR-4042: the upload failed because the file is larger than 50 MB.",
    "ERR-4043: the upload link has expired; request a new link.",
    "ERR-4044: you do not have permission to upload to this folder.",
    "Upload errors usually mean the network dropped; retry the upload.",
    "File size limits: free plans accept files up to 10 MB.",
]
q = "I keep getting ERR-4042"

bm25 = BM25Okapi([tokenize(d) for d in DOCS])        # 1. index the tokenized documents
scores = bm25.get_scores(tokenize(q))                # 2. score every document against the query
best = bm25.get_top_n(tokenize(q), DOCS, n=1)[0]     # 3. take the best one
print("BM25 scores :", np.round(scores, 2))
print("best match  :", best)
assert best == DOCS[1] and np.count_nonzero(scores) == 1       # only the exact code scores at all

naive = BM25Okapi([d.lower().split() for d in DOCS]).get_scores(q.lower().split())
print("\nwith str.split() the document token is 'err-4042:' (colon attached), so:", np.round(naive, 2))
assert not naive.any()                                          # the analyzer bug: nothing matches

Now the same query through a dense embedding model (all-MiniLM-L6-v2, a common default). This model is loaded here and reused in topic 9:

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = load_hf(SentenceTransformer, "sentence-transformers/all-MiniLM-L6-v2")
if embedder is not None:
    dense = embedder.encode(DOCS, normalize_embeddings=True) @ embedder.encode(q, normalize_embeddings=True)
    print("dense scores:", np.round(dense, 3))
    print("dense top-1 :", DOCS[int(np.argmax(dense))])
    c1, c2 = embedder.encode(["ERR-4041", "ERR-4042"], normalize_embeddings=True)
    print(f"cosine('ERR-4041', 'ERR-4042') = {c1 @ c2:.2f}: nearly the same point to the embedding model")
    assert int(np.argmax(dense)) != 1       # dense retrieval picks a look-alike code; BM25 picked the exact one

BM25 also gives diminishing returns for repeats. At average document length, a term's contribution is `f·(k1+1) / (f+k1)`, which can never exceed `k1 + 1`. Stuffing a keyword ten times buys almost nothing over saying it twice:

In [ ]:
tf = np.arange(0, 11)
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(tf, tf, color=GRAY, lw=1.5, label="raw count (no saturation)")
for k1, color in [(0.5, "#86b6ef"), (1.2, "#2a78d6"), (2.0, "#104281")]:     # ordered values: one hue, light -> dark
    ax.plot(tf, tf * (k1 + 1) / (tf + k1), marker="o", ms=4, color=color, label=f"BM25, k1 = {k1}")
ax.set_ylim(0, 4); ax.set_xlabel("times the term appears in the document"); ax.set_ylabel("contribution (x IDF)")
ax.set_title("Term-frequency saturation: the 10th mention adds almost nothing")
ax.legend(fontsize=8, loc="upper left"); plt.show()

contrib = tf * 2.2 / (tf + 1.2)
assert contrib[-1] < 2.2 and np.all(np.diff(np.diff(contrib)) < 0)   # bounded by k1+1, and each repeat adds less

<a id="t8"></a>
## 8 · KV cache

**In plain English.** An LLM writes one token at a time, and each new token attends to *all* earlier tokens through their keys (K) and values (V). Those never change once computed, so recomputing them for the whole prefix at every step wastes work. The **KV cache** stores them. Each step computes K and V for the new token only, appends them, and attends over the cache. Reading the cache still grows with the prefix, but it is far cheaper than re-projecting it. You trade memory for speed. The cache grows with context length × batch size, and at long context it, not the weights, often limits how many users one GPU can serve.

**Memory per token** = 2 (K and V) × layers × KV heads × head dim × bytes. For a Llama-3-8B-class model (32 layers, 8 KV heads thanks to grouped-query attention, head dim 128, fp16) that is 2 × 32 × 8 × 128 × 2 bytes = **128 KiB per token**, which is 1 GiB for an 8k-token conversation.

> **30-second answer.** "When decoding, each new token attends over all previous tokens' keys and values. Those don't change, so we cache them instead of recomputing, and each step projects only the new token. That turns quadratic recomputation into linear work. The cost is memory: 2 × layers × KV heads × head dim × bytes per token, about 128 KiB per token for an 8B model with GQA, or 1 GiB per sequence at 8k context. That is why GQA/MQA, KV-cache quantisation and PagedAttention exist. It also explains the latency profile: prefill (the first token) is slow, and each later token is fast."

**Deep dive:** [KV cache & inference speed](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb)

In [ ]:
rng = np.random.default_rng(8)
d, steps = 256, 256
Wq, Wk, Wv = (rng.normal(0, d ** -0.5, (d, d)).astype(np.float32) for _ in range(3))
X = rng.normal(0, 1, (steps, d)).astype(np.float32)          # the token embeddings, one per generated token

def attend(q, K, V):
    s = K @ q / np.sqrt(d)
    w = np.exp(s - s.max())
    return (w / w.sum()) @ V

def decode(use_cache):
    K_cache, V_cache = np.empty((steps, d), np.float32), np.empty((steps, d), np.float32)
    outs, elapsed, t0 = [], [], time.perf_counter()
    for t in range(1, steps + 1):
        if use_cache:            # project only the newest token and append it
            K_cache[t - 1], V_cache[t - 1] = X[t - 1] @ Wk, X[t - 1] @ Wv
            K, V = K_cache[:t], V_cache[:t]
        else:                    # recompute K and V for the whole prefix, every single step
            K, V = X[:t] @ Wk, X[:t] @ Wv
        outs.append(attend(X[t - 1] @ Wq, K, V))
        elapsed.append(time.perf_counter() - t0)
    return np.array(outs), np.array(elapsed) * 1000

out_slow, ms_slow = decode(use_cache=False)
out_fast, ms_fast = decode(use_cache=True)
print(f"recompute every step: {ms_slow[-1]:6.1f} ms for {steps} tokens")
print(f"with a KV cache     : {ms_fast[-1]:6.1f} ms  ({ms_slow[-1] / ms_fast[-1]:.1f}x faster)")
assert np.allclose(out_slow, out_fast, atol=1e-4)      # identical outputs: the cache changes speed, not results
assert ms_fast[-1] < ms_slow[-1]

In [ ]:
per_token_slow, per_token_fast = np.diff(ms_slow, prepend=0), np.diff(ms_fast, prepend=0)
rolling_median = lambda v, w=31: np.array([np.median(v[max(0, i - w):i + 1]) for i in range(len(v))])

fig, ax = plt.subplots(figsize=(8.5, 3.2))
ax.plot(np.arange(1, steps + 1), rolling_median(per_token_slow), label="recompute K, V every step")
ax.plot(np.arange(1, steps + 1), rolling_median(per_token_fast), label="KV cache")
ax.set_xlabel("position of the token being generated"); ax.set_ylabel("ms per token (rolling median)")
ax.set_title("Cost of each new token: grows with the prefix without a cache, stays flat with one")
ax.legend(); plt.show()

early, late = slice(0, 30), slice(-30, None)
print(f"no cache : first 30 tokens {np.median(per_token_slow[early]):.2f} ms each, last 30 {np.median(per_token_slow[late]):.2f} ms each")
print(f"KV cache : first 30 tokens {np.median(per_token_fast[early]):.2f} ms each, last 30 {np.median(per_token_fast[late]):.2f} ms each")
assert np.median(per_token_slow[late]) > 2 * np.median(per_token_slow[early])     # linear per step -> quadratic total
assert np.median(per_token_fast[late]) < np.median(per_token_slow[late])

In [ ]:
def kv_cache_bytes(layers, kv_heads, head_dim, seq_len, batch=1, bytes_per=2):
    return 2 * layers * kv_heads * head_dim * seq_len * batch * bytes_per

per_token = kv_cache_bytes(32, 8, 128, seq_len=1)
print(f"8B-class model, fp16: {per_token // 1024} KiB per token")
print(f"{'context':>9} | {'batch 1':>9} | {'batch 16':>9}")
for seq in [2_048, 8_192, 32_768, 131_072]:
    gib = [kv_cache_bytes(32, 8, 128, seq, batch) / 2**30 for batch in (1, 16)]
    print(f"{seq:>9,} | {gib[0]:6.2f} GiB | {gib[1]:6.1f} GiB")
without_gqa = kv_cache_bytes(32, 32, 128, 1)
print(f"without GQA (32 KV heads): {without_gqa // 1024} KiB per token, {without_gqa // per_token}x more")
assert per_token == 128 * 1024

<a id="t9"></a>
## 9 · Semantic chunking

**In plain English.** Fixed-size chunking cuts every N characters, often mid-thought. **Semantic chunking** embeds each sentence, compares each sentence with the next, and cuts where the similarity **drops**, which is where the topic changes. The threshold is a choice. Common rules are a percentile of the drops, mean minus k standard deviations, or the interquartile range, and LangChain's `SemanticChunker` offers exactly these. It costs one embedding per sentence at indexing time, and the benefit depends on the corpus. On well-structured docs, heading-aware or recursive splitting with overlap is a strong and cheaper baseline, so measure retrieval quality before paying for it.

> **30-second answer.** "Semantic chunking splits where the meaning changes rather than at a fixed length. I embed consecutive sentences and cut where cosine similarity falls below a threshold, such as mean minus one standard deviation or a percentile. Chunks stay on one topic, which helps retrieval precision. The costs are an embedding call per sentence at indexing, uneven chunk sizes (so cap the length), and a threshold to tune. It's worth it for long, unstructured text. For documents with headings, structure-aware splitting is usually as good and cheaper, and I decide by measuring recall@k on an eval set."

**Deep dive:** [Chunking strategies & semantic chunking](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb)

In [ ]:
SENTENCES = [
    "Refunds are issued to the original payment method.",
    "Most refunds reach your card within five business days.",
    "Annual plans can be refunded in full during the first thirty days.",
    "After that, refunds are prorated by the unused months.",
    "To reset your password, open Settings and choose Security.",
    "We email you a password reset link that expires after one hour.",
    "Your new password must be at least twelve characters long.",
    "For extra account security, turn on two-factor authentication.",
    "Orders ship from our warehouse in Rotterdam.",
    "Standard delivery of an order takes three to five working days.",
    "Express shipping arrives the next day if you order before noon.",
    "Every shipped order comes with a tracking number.",
]
TOPIC = ["refunds"] * 4 + ["passwords"] * 4 + ["shipping"] * 4      # ground truth, used only to check the result

if embedder is not None:                          # the MiniLM model loaded in topic 7
    E, backend = embedder.encode(SENTENCES, normalize_embeddings=True), "all-MiniLM-L6-v2"
else:
    E = TfidfVectorizer(stop_words="english").fit_transform(SENTENCES).toarray()
    E, backend = E / np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-12), "TF-IDF (fallback, lexical only)"

sims = np.sum(E[:-1] * E[1:], axis=1)           # cosine similarity of each sentence with the next one
threshold = sims.mean() - sims.std()            # the "standard deviation" breakpoint rule with k = 1
cuts = (np.where(sims < threshold)[0] + 1).tolist()
bounds = list(zip([0, *cuts], [*cuts, len(SENTENCES)]))
for i, (a, b) in enumerate(bounds, 1):
    print(f"chunk {i} ({b - a} sentences): {' '.join(SENTENCES[a:b])[:95]}...")
print(f"embeddings: {backend}")
if backend == "all-MiniLM-L6-v2":
    assert len(bounds) == 3 and all(len(set(TOPIC[a:b])) == 1 for a, b in bounds)   # 3 chunks, one topic each

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
pairs = np.arange(1, len(sims) + 1)
ax.plot(pairs, sims, marker="o", color=BLUE, label="similarity: sentence i vs sentence i+1")
ax.axhline(threshold, color=ORANGE, lw=1.5, label=f"threshold (mean - std) = {threshold:.2f}")
for c in cuts:
    ax.annotate(f"cut after\nsentence {c}", xy=(c, sims[c - 1]), xytext=(c + 0.9, threshold + 0.2), ha="center",
                fontsize=8, color=INK, arrowprops=dict(arrowstyle="->", color=INK))
ax.set_xticks(pairs); ax.set_xlabel("sentence i"); ax.set_ylabel("cosine similarity")
ax.set_title("Semantic chunking: start a new chunk where similarity drops below the threshold")
ax.legend(fontsize=8, loc="lower left"); plt.show()

<a id="t10"></a>
## 10 · Pydantic

**In plain English.** An LLM returns text, and your code needs a typed object. Pydantic is the bridge. You declare the shape once as a class. Pydantic **validates** incoming data, coercing safe cases such as `"42"` to `42`, and otherwise raises a precise `ValidationError`. The same class generates the **JSON Schema** that structured outputs and function calling send to the model. Three calls cover most LLM work: `model_json_schema()` gives the contract you send, `model_validate_json(text)` parses the reply, and `model_dump()` turns it back into a dict for logs and APIs. FastAPI uses the same models for request and response bodies.

> **30-second answer.** "Pydantic defines the output contract as a typed class. I send `Model.model_json_schema()` as the response format or tool schema, parse the reply with `Model.model_validate_json(reply)`, and on a `ValidationError` I retry with the error message fed back to the model. `extra='forbid'` puts `additionalProperties: false` in the schema, which OpenAI's strict structured outputs require. With the OpenAI SDK, `client.chat.completions.parse(..., response_format=Model)` does all of that and hands back `.parsed`."

**Deep dives:** [Type hints & Pydantic](../01_python_basics/13_type_hints_and_pydantic.ipynb) · [Structured outputs with Pydantic](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb)

In [ ]:
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError

class Ticket(BaseModel):
    model_config = ConfigDict(extra="forbid")          # unknown keys are errors -> "additionalProperties": false
    department: Literal["billing", "technical", "account"]
    urgent: bool
    summary: str = Field(description="one-line summary of the problem")
    order_id: int | None = None

reply = '{"department": "billing", "urgent": "true", "summary": "Charged twice", "order_id": "1042"}'
ticket = Ticket.model_validate_json(reply)
print("parsed :", repr(ticket))
print("dumped :", ticket.model_dump())
assert ticket.urgent is True and ticket.order_id == 1042        # safe coercion: "true" -> True, "1042" -> 1042

bad_reply = '{"department": "sales", "urgent": "maybe", "summary": "x", "priority": 1}'
try:
    Ticket.model_validate_json(bad_reply)
    raise AssertionError("expected a ValidationError")
except ValidationError as e:
    print(f"\n{e.error_count()} errors, precise enough to feed back to the model on a retry:")
    for err in e.errors():
        print(f"  {'.'.join(map(str, err['loc'])):11} {err['msg']}")
    assert e.error_count() == 3

In [ ]:
schema = Ticket.model_json_schema()        # this is what goes into response_format / a tool's "parameters"
print(json.dumps(schema, indent=1))
assert schema["additionalProperties"] is False
assert set(schema["required"]) == {"department", "urgent", "summary"}
assert schema["properties"]["department"]["enum"] == ["billing", "technical", "account"]

<a id="t11"></a>
## 11 · P95 latency

**In plain English.** p95 is the latency that 95% of requests beat, so one request in 20 is slower. The **mean hides the tail**. A few very slow requests (retries, cold starts, long outputs, a stuck tool call) barely move the average, yet they are what users complain about. Tails also compound. A page that makes 10 calls in a row hits at least one call at p95 or worse 1 − 0.95¹⁰ ≈ **40%** of the time. That is why SLOs and alerts are set on p95 and p99, and why "average latency" alone is a red flag in a design review.

> **30-second answer.** "p50 is the typical request. p95 and p99 are the tail, where 1 in 20 and 1 in 100 requests are slower. I set SLOs and alerts on p95/p99 because the mean hides the slow tail users feel, and fan-out makes the tail worse. For LLM calls I split latency into time-to-first-token and tokens per second, because streaming fixes perceived latency, not total time. To cut p95 I find what's in the tail (long outputs, retries, cold starts, a slow retriever) and fix that: shorter outputs, caching, timeouts, hedged requests, smaller models for easy requests."

**Deep dive:** [Latency percentiles p50/p95 & cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb)

In [ ]:
rng = np.random.default_rng(11)
latency = rng.lognormal(mean=np.log(800), sigma=0.35, size=2_000)      # typical LLM calls, ~0.8 s
slow = rng.random(latency.size) < 0.04                                 # 4% hit a retry or a cold start
latency[slow] += rng.uniform(2_000, 6_000, slow.sum())
p50, p95, p99 = np.percentile(latency, [50, 95, 99])
print(f"mean {latency.mean():5.0f} ms | p50 {p50:5.0f} ms | p95 {p95:5.0f} ms | p99 {p99:5.0f} ms")

fig, ax = plt.subplots(figsize=(9.5, 3.4))
ax.hist(latency, bins=90, color=BLUE, alpha=0.9)
for value, name, color in [(latency.mean(), "mean", GRAY), (p50, "p50", AQUA), (p95, "p95", ORANGE), (p99, "p99", "#e34948")]:
    ax.axvline(value, color=color, lw=1.8, label=f"{name} = {value:,.0f} ms")
ax.set_xlabel("latency (ms)"); ax.set_ylabel("requests")
ax.set_title("The mean sits in the body; p95/p99 show the slow tail users actually complain about")
ax.legend(fontsize=8); plt.show()

assert p50 < p95 < p99
assert latency.mean() > p50                                  # the tail drags the mean above the median
assert abs((latency > p95).mean() - 0.05) < 0.005            # by definition, 5% of requests are slower than p95

In [ ]:
calls_per_page = 10
analytic = 1 - 0.95 ** calls_per_page
pages = rng.choice(latency, size=(20_000, calls_per_page))               # simulate pages of 10 calls each
simulated = (pages > p95).any(axis=1).mean()
print(f"P(a {calls_per_page}-call page contains a call slower than p95): {analytic:.1%} (formula) vs {simulated:.1%} (simulated)")
assert abs(simulated - analytic) < 0.02

<a id="t12"></a>
## 12 · `np.random.seed` vs `np.random.default_rng`

**In plain English.** Computer "random" numbers are *pseudo*-random. A seed fixes the whole sequence, so a run can be repeated exactly. `np.random.seed(0)` is the **legacy** way. It seeds one hidden global generator that all code shares, so any library call that draws a random number shifts your sequence, and results depend on call order. The modern way is **`rng = np.random.default_rng(0)`**, an explicit generator object that you pass to the code that needs it. Streams stay independent, and the algorithm is better (PCG64 instead of the legacy Mersenne Twister). For ML reproducibility, seed every source (`random`, NumPy, `torch`). GPU kernels and data-loader workers add their own randomness.

> **30-second answer.** "`np.random.seed` sets hidden global state. It works, but any other code that draws from `np.random` shifts my stream. I use `rng = np.random.default_rng(seed)` and pass the generator explicitly. For experiments I seed `random`, NumPy and torch, and I report results over several seeds, because seed-to-seed variance can be as big as the improvement I'm claiming. For LLM APIs, `temperature=0` and the `seed` parameter are best-effort, not a guarantee."

**Deep dive:** [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb)

In [ ]:
import random

np.random.seed(0); a = np.random.rand(3)
np.random.seed(0); b = np.random.rand(3)
print("legacy global seed, run twice      :", np.allclose(a, b))

def some_library_helper():               # e.g. a data-augmentation function you didn't write
    return np.random.rand()              # silently draws from the SAME global generator

np.random.seed(0); some_library_helper(); c = np.random.rand(3)
print("after a hidden library draw        :", np.allclose(a, c), "<- your 'seeded' numbers changed")

mine, theirs = np.random.default_rng(0), np.random.default_rng(123)
x1 = mine.random(3)
mine = np.random.default_rng(0); theirs.random(); x2 = mine.random(3)
print("default_rng: other code can't touch:", np.allclose(x1, x2))
print("same seed 0, different algorithm   :", np.allclose(a, x1), "(legacy MT19937 vs PCG64)")
assert np.allclose(a, b) and not np.allclose(a, c) and np.allclose(x1, x2) and not np.allclose(a, x1)

In [ ]:
import torch
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def seed_everything(seed: int) -> np.random.Generator:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)     # torch.manual_seed covers CUDA too
    return np.random.default_rng(seed)

seed_everything(7); t1 = torch.rand(2)
seed_everything(7); t2 = torch.rand(2)
assert torch.equal(t1, t2)

# Why seeds matter for evals: same model, same data, only the random train/test split changes.
X, y = load_breast_cancer(return_X_y=True)
accs = []
for split_seed in range(20):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=split_seed)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
    accs.append(clf.score(X_test, y_test))
accs = np.array(accs)
print(f"accuracy over 20 split seeds: {accs.mean():.3f} +/- {accs.std():.3f} (range {accs.min():.3f} to {accs.max():.3f})")
assert accs.max() - accs.min() > 0.01            # a "1-point improvement" from one run can be pure seed noise

fig, ax = plt.subplots(figsize=(8, 2.4))
ax.scatter(accs, np.zeros_like(accs) + rng.uniform(-0.3, 0.3, accs.size), s=40, color=BLUE, alpha=0.8)
ax.axvline(accs.mean(), color=INK, lw=1); ax.set_yticks([]); ax.set_ylim(-1, 1)
ax.set_xlabel("test accuracy (one dot per random split)")
ax.set_title("Identical model, 20 seeds: the spread is the noise floor for any claimed improvement")
plt.show()

## Try it yourself

**1.** KV cache for a 70B-class model: 80 layers, 8 KV heads, head dim 128, fp16. How many GiB for a batch of 4 conversations at 32k context? Predict first, then use `kv_cache_bytes`.

In [ ]:
# Solution — try it yourself first, then run this
gib = kv_cache_bytes(layers=80, kv_heads=8, head_dim=128, seq_len=32_768, batch=4) / 2**30
print(f"70B-class, 32k context, batch 4: {gib:.0f} GiB of KV cache (plus ~140 GB of fp16 weights)")
assert gib == 40

**2.** Quantize `W` to **int4** with per-row absmax (levels −7..7) and compare the reconstruction error with int8. Predict how much worse it is.

In [ ]:
# Solution — try it yourself first, then run this
def absmax_roundtrip(w, bits):
    qmax = 2 ** (bits - 1) - 1                                    # 127 for int8, 7 for int4
    scale = np.abs(w).max(axis=1, keepdims=True) / qmax
    return np.clip(np.round(w / scale), -qmax, qmax) * scale

errs = {bits: np.linalg.norm(W - absmax_roundtrip(W, bits)) / np.linalg.norm(W) for bits in (8, 4)}
print(f"int8 error {errs[8]:.2%} | int4 error {errs[4]:.2%} ({errs[4] / errs[8]:.0f}x worse; 16x fewer levels)")
assert errs[4] > 10 * errs[8]

**3.** Add a `confidence: float` field to `Ticket` that must lie in [0, 1], and show that `1.3` is rejected.

In [ ]:
# Solution — try it yourself first, then run this
class ScoredTicket(Ticket):
    confidence: float = Field(ge=0, le=1)

try:
    ScoredTicket(department="billing", urgent=False, summary="ok", confidence=1.3)
    raise AssertionError("1.3 should have been rejected")
except ValidationError as e:
    print("rejected:", e.errors()[0]["msg"])
print("accepted:", ScoredTicket(department="billing", urgent=False, summary="ok", confidence=0.9).confidence)

**4.** A page makes 5 calls. What fraction of pages include at least one call slower than **p99**? Work out the formula, then check it by simulation with the `latency` sample above.

In [ ]:
# Solution — try it yourself first, then run this
formula = 1 - 0.99 ** 5
sim = (rng.choice(latency, size=(50_000, 5)) > p99).any(axis=1).mean()
print(f"formula {formula:.1%} vs simulated {sim:.1%}: a 'one in a hundred' tail touches ~1 page in 20")
assert abs(sim - formula) < 0.01

## Say it in an interview

| Topic | The one-liner | The trap they test |
|---|---|---|
| Jev & Laya | typed questions in, calibrated probabilities out, no text to parse; escalate below a confidence threshold | an LLM plus regex for every routing decision |
| Power BI Python visual | Power BI passes a de-duplicated `dataset` DataFrame; you draw with matplotlib and `plt.show()` | identical rows silently merged: add an ID field |
| Overfitting | training error falls, validation error rises; fix with data, simplicity, regularisation, early stopping | judging a model on training accuracy |
| Quantization | int8 = 4× smaller than fp32; a 7B model is 28 GB → 3.5 GB at int4 | forgetting the KV cache and activations in the memory budget |
| LiteLLM | one `completion()` for 100+ providers; Router fallbacks; `mock_response` for tests | fallback paths that were never tested |
| Re-ranking | cheap top-k, a cross-encoder reorders, keep the top 3–5 | expecting a reranker to fix recall |
| BM25 | IDF × saturating TF × length normalisation; exact IDs win | a tokenizer that glues punctuation onto codes |
| KV cache | cache K and V; 2·layers·KV heads·head dim·bytes ≈ 128 KiB per token for an 8B model with GQA | forgetting it scales with batch × context |
| Semantic chunking | cut where consecutive-sentence similarity drops | paying for it where heading-aware splitting is just as good |
| Pydantic | schema out (`model_json_schema`), validation in (`model_validate_json`) | not feeding the validation error back on a retry |
| P95 | 1 in 20 requests is slower; the mean hides it; fan-out amplifies it | reporting only the mean |
| Seeds | pass `default_rng(seed)` around; seed random, NumPy and torch; report results over several seeds | global `np.random.seed` state consumed by other code |

**Numbers to have ready:** 128 KiB per token of KV cache (8B, GQA, fp16) · 7B weights: 28 / 14 / 7 / 3.5 GB at fp32 / fp16 / int8 / int4 · 1 − 0.95¹⁰ ≈ 40% · BM25 `k1` ≈ 1.2–2, `b` = 0.75 · rerank the top 20–100, keep the top 3–5.

## Next

- Deep dives, in the order above: [System-1 models: Jev & Laya](../15_agentic_ai/08_system1_decision_models_jev_and_laya.ipynb) · [matplotlib & quick charts](../04_numpy_pandas_pytorch/04_matplotlib_and_quick_charts.ipynb) · [overfitting & CV](../05_machine_learning/05_overfitting_regularization_cross_validation.ipynb) · [quantization](../07_genai_foundations/07_quantization_int8_int4_nf4.ipynb) · [bitsandbytes](../10_huggingface/06_quantization_with_bitsandbytes.ipynb) · [LiteLLM](../16_production/06_litellm_gateway_and_model_routing.ipynb) · [reranking](../09_rag/05_reranking.ipynb) · [BM25](../09_rag/02_bm25_from_scratch.ipynb) · [KV cache](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb) · [chunking](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb) · [Pydantic](../01_python_basics/13_type_hints_and_pydantic.ipynb) · [structured outputs](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb) · [p95 & cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb) · [NumPy](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb)
- Related: [hybrid search & RRF](../09_rag/04_hybrid_search_and_rrf.ipynb) combines topics 6 and 7.
- Theory: [interview bank](../../ai_interview_prep/index.html) · [genai_flow course](../../genai_flow/index.html) · [AI system design concepts](../../ai_system_design_concepts/index.html)